In [7]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import json, os, warnings

# Suppressing TensorFlow logging and warnings for a cleaner output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import xgboost as xgb
import shap
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix

# Ensuring reproducible results
np.random.seed(42)

# Loading the engineered dataset and feature definitions
df = pd.read_csv('../data/processed/FD001_with_rul.csv')
with open('../data/processed/useful_sensors.json') as f:
    useful_sensors = json.load(f)
with open('../data/processed/feature_cols.json') as f:
    feature_cols = json.load(f)

# Reconstructing the exact train/test split used in previous modeling steps
engines      = df['unit_id'].unique()
test_engines = np.random.choice(engines, size=int(0.2 * len(engines)), replace=False)
test_mask    = df['unit_id'].isin(test_engines)
train_mask   = ~test_mask

# Defining the business logic for alert zones
def rul_to_alert(rul):
    if rul <= 30:   return 'CRITICAL'
    elif rul <= 60: return 'WARNING'
    else:           return 'NORMAL'

# Applying alert labels to the dataset
df['alert'] = df['RUL'].apply(rul_to_alert)
print(f"Data loaded successfully. Number of test engines: {len(test_engines)}")

Data loaded successfully. Number of test engines: 20


In [2]:
# Generating XGBoost predictions on the testing set
xgb_model = xgb.XGBRegressor()
xgb_model.load_model('../results/models/xgboost.json')

df_test = df[test_mask].copy()
df_test['RUL_pred_xgb'] = xgb_model.predict(df_test[feature_cols])

# Preparing sequences and generating LSTM predictions
WINDOW   = 30
FEATURES = useful_sensors + ['op_setting_1', 'op_setting_2', 'op_setting_3']
scaler   = MinMaxScaler()
df_scaled = df.copy()
df_scaled[FEATURES] = scaler.fit_transform(df[FEATURES])
lstm_model = tf.keras.models.load_model('../results/models/lstm_rul.keras')

lstm_preds = {}
for eng in test_engines:
    group = df_scaled[df_scaled['unit_id'] == eng].sort_values('cycle')
    feats = group[FEATURES].values
    preds = [np.nan] * WINDOW
    
    # Simulating real-time sequential prediction
    for i in range(WINDOW, len(feats)):
        seq = feats[i-WINDOW:i][np.newaxis]
        preds.append(float(lstm_model.predict(seq, verbose=0)[0][0]))
    lstm_preds[eng] = preds

# Selecting 6 random engines for visualization
sample_engines = np.random.choice(list(test_engines), size=6, replace=False)

# Constructing interactive subplots for degradation curves
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'Engine ID: {e}' for e in sample_engines],
    shared_yaxes=False
)

alert_color = {'NORMAL': '#2a9d8f', 'WARNING': '#e9c46a', 'CRITICAL': '#e63946'}

for idx, eng in enumerate(sample_engines):
    row = idx // 3 + 1
    col = idx %  3 + 1
    grp = df[df['unit_id'] == eng].sort_values('cycle')
    grp_test = df_test[df_test['unit_id'] == eng].sort_values('cycle')

    # Plotting the Ground Truth RUL
    fig.add_trace(go.Scatter(
        x=grp['cycle'], y=grp['RUL'],
        mode='lines', name='True RUL',
        line=dict(color='#457b9d', width=2),
        showlegend=(idx == 0)
    ), row=row, col=col)

    # Plotting XGBoost Predictions
    fig.add_trace(go.Scatter(
        x=grp_test['cycle'], y=grp_test['RUL_pred_xgb'],
        mode='lines', name='XGBoost',
        line=dict(color='#2a9d8f', width=1.5, dash='dash'),
        showlegend=(idx == 0)
    ), row=row, col=col)

    # Plotting LSTM Predictions
    lstm_p = lstm_preds[eng]
    cycles = grp['cycle'].values
    fig.add_trace(go.Scatter(
        x=cycles, y=lstm_p,
        mode='lines', name='LSTM',
        line=dict(color='#e63946', width=1.5, dash='dot'),
        showlegend=(idx == 0)
    ), row=row, col=col)

    # Adding critical and warning threshold lines
    for thresh, color in [(30, '#e63946'), (60, '#e9c46a')]:
        fig.add_hline(y=thresh, line_dash='dot', line_color=color,
                      opacity=0.5, row=row, col=col)

# Formatting and exporting the interactive plot
fig.update_layout(
    title='Engine Degradation Curves — Predicted vs Actual RUL',
    height=600, template='plotly_dark',
    font=dict(size=11)
)
fig.write_html('../results/degradation_curves.html')
fig.show()

In [3]:
# Loading and visualizing the final benchmark leaderboard
bm = pd.read_csv('../results/benchmark_table.csv').sort_values('RMSE').reset_index(drop=True)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['RMSE', 'MAE', 'NASA Score']
)

palette = px.colors.qualitative.Bold

for i, metric in enumerate(['RMSE', 'MAE', 'NASA_Score']):
    fig.add_trace(go.Bar(
        x=bm['Model'], y=bm[metric],
        marker_color=palette[:len(bm)],
        text=bm[metric].round(2),
        textposition='outside',
        showlegend=False
    ), row=1, col=i+1)

fig.update_layout(
    title='Comprehensive Model Benchmark — AeroGuard Project (FD001)',
    height=450, template='plotly_dark',
    font=dict(size=12)
)
fig.write_html('../results/benchmark_chart.html')
fig.show()

In [4]:
# Extracting and visualizing global SHAP values using Plotly for interactivity
X_test_xgb  = df[test_mask][feature_cols]
explainer   = shap.TreeExplainer(xgb_model)

# Sampling data to optimize SHAP computation speed
sample_idx  = np.random.choice(len(X_test_xgb), size=500, replace=False)
X_sample    = X_test_xgb.iloc[sample_idx]
shap_values = explainer(X_sample)

# Aggregating absolute SHAP values
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_df = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=True).tail(20)

fig = go.Figure(go.Bar(
    x=shap_df['mean_abs_shap'],
    y=shap_df['feature'],
    orientation='h',
    marker=dict(
        color=shap_df['mean_abs_shap'],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title='SHAP Impact')
    )
))

fig.update_layout(
    title='SHAP Global Feature Importance — Top 20 (XGBoost)',
    xaxis_title='Mean |SHAP Value| (Impact on Model Output)',
    height=600, template='plotly_dark',
    font=dict(size=11)
)
fig.write_html('../results/shap_summary_plotly.html')
fig.show()

In [5]:
# Generating an interactive heatmap for the Early Warning System Confusion Matrix
LABEL_NAMES = ['NORMAL', 'WARNING', 'CRITICAL']

def alert_int(rul):
    if rul <= 30:   return 2
    elif rul <= 60: return 1
    else:           return 0

y_true = df[test_mask]['RUL'].apply(alert_int).values
y_pred = np.array([alert_int(r) for r in xgb_model.predict(df[test_mask][feature_cols])])

# Calculating matrix values and percentages
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

text = [[f'{cm[i][j]}<br>({cm_pct[i][j]:.1f}%)' for j in range(3)] for i in range(3)]

fig = go.Figure(go.Heatmap(
    z=cm_pct,
    x=LABEL_NAMES,
    y=LABEL_NAMES,
    text=text,
    texttemplate='%{text}',
    colorscale='Reds',
    showscale=True
))

fig.update_layout(
    title='XGBoost — Early Warning System Confusion Matrix',
    xaxis_title='Predicted Alert Zone',
    yaxis_title='True Alert Zone',
    height=450, template='plotly_dark',
    font=dict(size=12)
)
fig.write_html('../results/confusion_matrix_plotly.html')
fig.show()

In [6]:
# Simulating a real-time fleet monitoring dashboard by plotting the last cycle of each engine
last_cycles = df[test_mask].sort_values('cycle').groupby('unit_id').last().reset_index()
last_cycles['RUL_pred'] = xgb_model.predict(last_cycles[feature_cols])
last_cycles['alert_pred'] = last_cycles['RUL_pred'].apply(rul_to_alert)
last_cycles['alert_true'] = last_cycles['RUL'].apply(rul_to_alert)

color_map = {'NORMAL': '#2a9d8f', 'WARNING': '#e9c46a', 'CRITICAL': '#e63946'}

fig = go.Figure()

for alert in ['NORMAL', 'WARNING', 'CRITICAL']:
    sub = last_cycles[last_cycles['alert_pred'] == alert]
    fig.add_trace(go.Scatter(
        x=sub['RUL'],
        y=sub['RUL_pred'],
        mode='markers',
        name=alert,
        marker=dict(color=color_map[alert], size=10, opacity=0.85,
                    line=dict(width=1, color='white')),
        text=sub['unit_id'].apply(lambda x: f'Engine ID: {x}'),
        hovertemplate='%{text}<br>True RUL: %{x}<br>Predicted RUL: %{y}<extra></extra>'
    ))

# Plotting the ideal perfect prediction diagonal
max_rul = int(last_cycles['RUL'].max()) + 10
fig.add_trace(go.Scatter(
    x=[0, max_rul], y=[0, max_rul],
    mode='lines', name='Perfect Prediction',
    line=dict(color='white', dash='dash', width=1),
    opacity=0.4
))

# Annotating critical and warning thresholds
fig.add_vline(x=30, line_dash='dot', line_color='#e63946', opacity=0.6, annotation_text='Critical Threshold (30)')
fig.add_vline(x=60, line_dash='dot', line_color='#e9c46a', opacity=0.6, annotation_text='Warning Threshold (60)')

fig.update_layout(
    title='Fleet Overview Dashboard — Final Cycle Snapshot: True vs Predicted RUL',
    xaxis_title='True RUL (Cycles Remaining)',
    yaxis_title='Predicted RUL',
    height=550, template='plotly_dark',
    font=dict(size=12)
)
fig.write_html('../results/fleet_overview.html')
fig.show()